# YOLOv8 Dental Detection: Tooth Types + Damage

This notebook trains a YOLOv8 model to detect:
- **Tooth Types**: Incisors, Canines, Premolars, Molars
- **Damage**: Caries, Cavity, Decay, Crack
- **Position**: Upper/Lower (calculated from coordinates)

## Dataset Requirements

Your dataset must have these classes:
- `incisor`, `canine`, `premolar`, `molar` (tooth types)
- `caries`, `cavity`, `decay`, `crack` (damage types)

## Notebook Structure
1. Dataset Preparation
2. Model Training
3. Inference & Visualization
4. Model Evaluation
5. Application Integration

---
## Part 1: Dataset Preparation

### Option A: Using Roboflow (Easiest)
Uncomment and use this if you have a Roboflow dataset

In [1]:
# # Install Roboflow
# !pip install roboflow

# from roboflow import Roboflow

# # Initialize Roboflow
# rf = Roboflow(api_key="YOUR_API_KEY")
# project = rf.workspace().project("YOUR_PROJECT_NAME")
# dataset = project.version(1).download("yolov8")

# print(f"Dataset downloaded to: {dataset.location}")

### Option B: Manual Dataset Conversion
Use this if you have a custom dataset to convert to YOLO format

In [14]:
import shutil
from os import path
import os
import json


# ============================================
# CONFIGURATION - UPDATE THESE PATHS
# ============================================
SRC_DIR = "./dentalai-DatasetNinja"  # YOUR SOURCE DATASET PATH
DEST_DIR = "yolo_dataset"  # Output directory

# ============================================
# CLASS DEFINITIONS - CRITICAL!
# ============================================
# Define your classes and their IDs
classes = {
    # Tooth types (0-3)
    'incisor': 0,
    'canine': 1,
    'premolar': 2,
    'molar': 3,
    
    # Damage types (4-7)
    'caries': 4,
    'cavity': 5,
    'decay': 6,
    'crack': 7,
}

# ============================================
# STEP 1: Create Folder Structure
# ============================================
print("Creating YOLO dataset structure...")
os.makedirs(path.join(DEST_DIR, "train", "images"), exist_ok=True)
os.makedirs(path.join(DEST_DIR, "train", "labels"), exist_ok=True)
os.makedirs(path.join(DEST_DIR, "val", "images"), exist_ok=True)
os.makedirs(path.join(DEST_DIR, "val", "labels"), exist_ok=True)
os.makedirs(path.join(DEST_DIR, "test", "images"), exist_ok=True)
os.makedirs(path.join(DEST_DIR, "test", "labels"), exist_ok=True)

# ============================================
# STEP 2: Create data.yaml file
# ============================================
print("Creating data.yaml configuration...")
with open(path.join(DEST_DIR, "data.yaml"), "w") as fp:
    fp.write("train: ../train/images\n")
    fp.write("val: ../valid/images\n")
    fp.write("test: ../test/images\n")
    fp.write("\n")
    fp.write(f"nc: {len(classes)}\n")
    fp.write(f"names: {list(classes.keys())}\n")

print(f"\nConfigured {len(classes)} classes:")
for name, idx in classes.items():
    print(f"  {idx}: {name}")

# ============================================
# STEP 3: Copy Images and Convert Annotations
# ============================================
print("\nCopying images and converting annotations...")

# If your source has 'valid' instead of 'val', adjust this mapping
dirs_map = {"train": "train", "valid": "val", "test": "test"}

for src_dir, dest_dir in dirs_map.items():
    src_path = path.join(SRC_DIR, src_dir)
    
    # Skip if source directory doesn't exist
    if not path.exists(src_path):
        print(f"⚠️  Skipping {src_dir} - directory not found")
        continue
    
    # Copy all images
    print(f"Processing {src_dir}...")
    shutil.copytree(path.join(src_path, "img"),
                    path.join(DEST_DIR, dest_dir, "images"),
                    dirs_exist_ok=True)
    
    # Convert annotations from JSON to YOLO format
    ann_dir = path.join(src_path, "ann")
    if path.exists(ann_dir):
        for file in os.listdir(ann_dir):
            if not file.endswith('.json'):
                continue
                
            ann = json.load(open(path.join(ann_dir, file), "r"))
            
            # Get image dimensions
            img_width = ann["size"]["width"]
            img_height = ann["size"]["height"]
            
            # Create corresponding .txt file
            file_name = file.replace(".jpg.json", ".txt")
            label_path = path.join(DEST_DIR, dest_dir, "labels", file_name)
            
            with open(label_path, "w") as fp:
                for obj in ann["objects"]:
                    class_title = obj["classTitle"]
                    
                    # Skip if class not in our defined classes
                    if class_title not in classes:
                        print(f"⚠️  Unknown class '{class_title}' in {file} - skipping")
                        continue
                    
                    class_id = classes[class_title]
                    
                    # Calculate bounding box from polygon points
                    top = float('inf')
                    left = float('inf')
                    bottom = float('-inf')
                    right = float('-inf')

                    for point in obj["points"]["exterior"]:
                        x, y = point
                        left = min(left, x)
                        right = max(right, x)
                        top = min(top, y)
                        bottom = max(bottom, y)

                    # Calculate width and height
                    width = right - left
                    height = bottom - top
                    
                    # Convert to YOLO format (normalized center coordinates)
                    x_center = (left + width / 2) / img_width
                    y_center = (top + height / 2) / img_height
                    norm_width = width / img_width
                    norm_height = height / img_height
                    
                    # Write to file: class_id x_center y_center width height
                    fp.write(f"{class_id} {x_center} {y_center} {norm_width} {norm_height}\n")

print("\n✅ Dataset conversion completed!")

# ============================================
# STEP 4: Verify Dataset
# ============================================
print("\n" + "="*50)
print("DATASET SUMMARY")
print("="*50)

for split in ['train', 'val', 'test']:
    img_dir = path.join(DEST_DIR, split, "images")
    lbl_dir = path.join(DEST_DIR, split, "labels")
    
    if path.exists(img_dir):
        img_count = len(os.listdir(img_dir))
        lbl_count = len(os.listdir(lbl_dir)) if path.exists(lbl_dir) else 0
        print(f"{split.upper():6} : {img_count:4} images, {lbl_count:4} labels")
    else:
        print(f"{split.upper():6} : Not found")

print("\n📁 Dataset location:", path.abspath(DEST_DIR))
print("📄 Config file:", path.join(DEST_DIR, "data.yaml"))

Creating YOLO dataset structure...
Creating data.yaml configuration...

Configured 8 classes:
  0: incisor
  1: canine
  2: premolar
  3: molar
  4: caries
  5: cavity
  6: decay
  7: crack

Copying images and converting annotations...
Processing train...
⚠️  Unknown class 'Tooth' in 1000_jpg.rf.ad94534c8a4bf33d828b910160011dd9.jpg.json - skipping
⚠️  Unknown class 'Tooth' in 1000_jpg.rf.ad94534c8a4bf33d828b910160011dd9.jpg.json - skipping
⚠️  Unknown class 'Tooth' in 1000_jpg.rf.ad94534c8a4bf33d828b910160011dd9.jpg.json - skipping
⚠️  Unknown class 'Tooth' in 1000_jpg.rf.ad94534c8a4bf33d828b910160011dd9.jpg.json - skipping
⚠️  Unknown class 'Tooth' in 1000_jpg.rf.ad94534c8a4bf33d828b910160011dd9.jpg.json - skipping
⚠️  Unknown class 'Tooth' in 1000_jpg.rf.ad94534c8a4bf33d828b910160011dd9.jpg.json - skipping
⚠️  Unknown class 'Tooth' in 1000_jpg.rf.ad94534c8a4bf33d828b910160011dd9.jpg.json - skipping
⚠️  Unknown class 'Tooth' in 1000_jpg.rf.ad94534c8a4bf33d828b910160011dd9.jpg.json - s

### Option C: If Your Data is Already in YOLO Format
Just verify your data.yaml has the correct classes

In [ ]:
# If your data is already in YOLO format, just set the path
# DEST_DIR = "./yolo-dataset"

# # Verify data.yaml exists
# import yaml
# with open(f"{DEST_DIR}/data.yaml", 'r') as f:
#     config = yaml.safe_load(f)
#     print("Dataset configuration:")
#     print(f"  Classes ({config['nc']}): {config['names']}")
#     print(f"  Train: {config['train']}")
#     print(f"  Val: {config['val']}")

FileNotFoundError: [Errno 2] No such file or directory: './yolo-dataset/data.yaml'

---
## Part 2: Install Dependencies

In [20]:
%pip install ultralytics wandb pillow matplotlib numpy nbformat


Defaulting to user installation because normal site-packages is not writeable
  Using cached nbformat-5.10.4-py3-none-any.whl.metadata (3.6 kB)
  Using cached fastjsonschema-2.21.2-py3-none-any.whl.metadata (2.3 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached rpds_py-0.30.0-cp314-cp314-win_amd64.whl.metadata (4.2 kB)
Using cached nbformat-5.10.4-py3-none-any.whl (78 kB)
Using cached fastjsonschema-2.21.2-py3-none-any.whl (24 kB)
Using cached attrs-25.4.0-py3-none-any.whl (67 kB)
Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl (18 kB)
Using cached referencing-0.37.0-py3-none-any.whl (26 kB)
Using cached rpds_py-0.30.0-cp314-cp314-win_amd64.whl (228 kB)

   ----------- ---------------------------- 2/7 [attrs]
   --------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## Part 3: Train the Model

In [ ]:
from ultralytics.models.yolo import YOLO
import os
import wandb

# ============================================
# CONFIGURATION
# ============================================

# Path to your data.yaml file
DATA_YAML = "yolo_dataset/data.yaml"  # Update if different

# Training hyperparameters
CONFIG = {
    # Model selection: 'yolov8n.pt', 'yolov8s.pt', 'yolov8m.pt', 'yolov8l.pt', 'yolov8x.pt'
    # n=nano (fastest), s=small, m=medium, l=large, x=extra-large (most accurate)
    "model": "yolov8s.pt",  # Good balance of speed and accuracy
    
    # Training parameters
    "epochs": 100,  # Number of training epochs (increase for better results)
    "batch_size": 16,  # Reduce if you get out-of-memory errors
    "img_size": 640,  # Input image size
    "patience": 20,  # Early stopping patience
    
    # Learning rate
    "lr0": 0.001,  # Initial learning rate
    
    # Device
    "device": CPU,  # 0 for GPU, 'cpu' for CPU
}

# ============================================
# INITIALIZE EXPERIMENT TRACKING (OPTIONAL)
# ============================================
# Comment out if you don't want to use Weights & Biases

wandb.init(
    project="Dental_Detection",
    name="YOLOv8_Tooth_Types_and_Damage",
    config=CONFIG
)

# ============================================
# LOAD PRE-TRAINED MODEL
# ============================================
print(f"Loading pre-trained model: {CONFIG['model']}")
model = YOLO(CONFIG["model"])

print(f"\nModel info:")
print(f"  Parameters: {sum(p.numel() for p in model.model.parameters()):,}")
print(f"  Layers: {len(list(model.model.modules()))}")

# ============================================
# TRAIN THE MODEL
# ============================================
print(f"\n{'='*60}")
print("STARTING TRAINING")
print(f"{'='*60}\n")

results = model.train(
    # Dataset
    data=DATA_YAML,
    
    # Training parameters
    epochs=CONFIG["epochs"],
    batch=CONFIG["batch_size"],
    imgsz=CONFIG["img_size"],
    patience=CONFIG["patience"],
    
    # Save settings
    project="Dental_Detection",
    name="YOLOv8_Training",
    save_period=10,  # Save checkpoint every 10 epochs
    
    # Data augmentation (helps prevent overfitting)
    hsv_h=0.015,      # Hue augmentation
    hsv_s=0.7,        # Saturation augmentation
    hsv_v=0.4,        # Value (brightness) augmentation
    degrees=10.0,     # Rotation augmentation (±10 degrees)
    translate=0.1,    # Translation augmentation
    scale=0.3,        # Scaling augmentation
    flipud=0.5,       # Vertical flip (50% chance)
    fliplr=0.5,       # Horizontal flip (50% chance)
    mosaic=0.8,       # Mosaic augmentation
    mixup=0.1,        # Mixup augmentation
    
    # Optimizer
    optimizer='AdamW',  # AdamW works well for small datasets
    lr0=CONFIG["lr0"],
    weight_decay=0.0005,
    
    # Device
    device=CONFIG["device"],
    
    # Multi-GPU (if available)
    # workers=8,  # Number of data loading workers
    
    # Validation
    val=True,  # Validate during training
)

print(f"\n{'='*60}")
print("TRAINING COMPLETED!")
print(f"{'='*60}\n")

# ============================================
# SAVE THE TRAINED MODEL
# ============================================
MODEL_SAVE_PATH = "dental_detection_model.pt"
model.save(MODEL_SAVE_PATH)
print(f"✅ Model saved to: {MODEL_SAVE_PATH}")

# Log to W&B
if wandb.run is not None:
    wandb.log({"model_path": MODEL_SAVE_PATH})
    wandb.finish()

# ============================================
# PRINT FINAL METRICS
# ============================================
print("\nFinal Training Results:")
print(f"  Best mAP50: {results.results_dict.get('metrics/mAP50(B)', 'N/A')}")
print(f"  Best mAP50-95: {results.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")
print(f"\nModel weights saved at:")
print(f"  Best: Dental_Detection/YOLOv8_Training/weights/best.pt")
print(f"  Last: Dental_Detection/YOLOv8_Training/weights/last.pt")

Loading pre-trained model: yolov8s.pt

Model info:
  Parameters: 11,166,560
  Layers: 225

STARTING TRAINING

Ultralytics 8.4.12  Python-3.14.2 torch-2.10.0+cpu 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 0
os.environ['CUDA_VISIBLE_DEVICES']: 0
See https://pytorch.org/get-started/locally/ for up-to-date torch install instructions if no CUDA devices are seen by torch.


---
## Part 4: Inference - DentalDetector Class

This class provides:
- Tooth and damage detection
- Position determination (upper/lower, left/right)
- Visualization
- Patient report generation

In [ ]:
from ultralytics import YOLO
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import numpy as np

class DentalDetector:
    """
    Dental Detection System for Tooth Types and Damage
    
    Detects:
    - Tooth types: incisors, canines, premolars, molars
    - Damage: caries, cavity, decay, crack
    - Position: upper/lower, left/right
    """
    
    def __init__(self, model_path):
        """
        Initialize the detector
        
        Args:
            model_path: Path to trained YOLOv8 model (.pt file)
        """
        print(f"Loading model from: {model_path}")
        self.model = YOLO(model_path)
        
        # Define class categories
        self.tooth_classes = ['incisor', 'canine', 'premolar', 'molar']
        self.damage_classes = ['caries', 'cavity', 'decay', 'crack']
        
        print("✅ Model loaded successfully!")
        print(f"   Tooth classes: {self.tooth_classes}")
        print(f"   Damage classes: {self.damage_classes}")
    
    def determine_position(self, bbox, img_height):
        """
        Determine if detection is in upper or lower jaw
        
        Args:
            bbox: [x1, y1, x2, y2] bounding box coordinates
            img_height: Height of the image
            
        Returns:
            'upper' or 'lower'
        """
        y_center = (bbox[1] + bbox[3]) / 2
        return 'upper' if y_center < img_height / 2 else 'lower'
    
    def determine_side(self, bbox, img_width):
        """
        Determine if detection is on left or right side
        
        Args:
            bbox: [x1, y1, x2, y2] bounding box coordinates
            img_width: Width of the image
            
        Returns:
            'left' or 'right'
        """
        x_center = (bbox[0] + bbox[2]) / 2
        return 'left' if x_center < img_width / 2 else 'right'
    
    def detect(self, image_path, conf_threshold=0.5, iou_threshold=0.45):
        """
        Perform detection on an image
        
        Args:
            image_path: Path to the input image
            conf_threshold: Confidence threshold (0-1)
            iou_threshold: IoU threshold for NMS
            
        Returns:
            dict with 'teeth' and 'damages' detections
        """
        # Run inference
        results = self.model(
            image_path,
            conf=conf_threshold,
            iou=iou_threshold,
            verbose=False
        )
        
        # Get image dimensions
        img = Image.open(image_path)
        img_width, img_height = img.size
        
        # Organize detections
        teeth = []
        damages = []
        
        for result in results:
            boxes = result.boxes
            
            for box in boxes:
                # Extract detection info
                class_id = int(box.cls.cpu().numpy())
                class_name = result.names[class_id]
                confidence = float(box.conf.cpu().numpy())
                bbox = box.xyxy[0].cpu().numpy().tolist()  # [x1, y1, x2, y2]
                
                # Determine position
                position = self.determine_position(bbox, img_height)
                side = self.determine_side(bbox, img_width)
                
                # Create detection object
                detection = {
                    'class': class_name,
                    'class_id': class_id,
                    'confidence': confidence,
                    'bbox': bbox,
                    'position': position,
                    'side': side,
                    'full_label': f"{position} {side} {class_name}"
                }
                
                # Categorize detection
                if class_name in self.tooth_classes:
                    teeth.append(detection)
                elif class_name in self.damage_classes:
                    damages.append(detection)
        
        return {
            'teeth': teeth,
            'damages': damages,
            'image_path': image_path,
            'image_size': (img_width, img_height)
        }
    
    def visualize(self, image_path, detections, save_path=None):
        """
        Visualize detections on the image
        
        Args:
            image_path: Path to the image
            detections: Output from detect() method
            save_path: Optional path to save the visualization
            
        Returns:
            PIL Image with visualizations
        """
        # Load image
        img = Image.open(image_path).convert("RGB")
        draw = ImageDraw.Draw(img)
        
        # Try to load a nice font
        try:
            font = ImageFont.truetype("arial.ttf", 16)
            font_large = ImageFont.truetype("arial.ttf", 20)
        except:
            font = ImageFont.load_default()
            font_large = ImageFont.load_default()
        
        # Draw teeth (green boxes)
        for tooth in detections['teeth']:
            bbox = tooth['bbox']
            
            # Draw bounding box
            draw.rectangle(bbox, outline="#00FF00", width=3)
            
            # Draw label background
            label = f"{tooth['full_label']} {tooth['confidence']:.0%}"
            text_bbox = draw.textbbox((bbox[0], bbox[1] - 25), label, font=font)
            draw.rectangle(text_bbox, fill="#00FF00")
            
            # Draw label text
            draw.text((bbox[0], bbox[1] - 25), label, fill="#000000", font=font)
        
        # Draw damages (red boxes)
        for damage in detections['damages']:
            bbox = damage['bbox']
            
            # Draw bounding box
            draw.rectangle(bbox, outline="#FF0000", width=3)
            
            # Draw label background
            label = f"{damage['class'].upper()} {damage['confidence']:.0%}"
            text_bbox = draw.textbbox((bbox[0], bbox[1] - 25), label, font=font)
            draw.rectangle(text_bbox, fill="#FF0000")
            
            # Draw label text
            draw.text((bbox[0], bbox[1] - 25), label, fill="#FFFFFF", font=font)
        
        # Display with matplotlib
        plt.figure(figsize=(15, 10))
        plt.imshow(img)
        plt.axis('off')
        plt.title(
            f"Detected: {len(detections['teeth'])} teeth, {len(detections['damages'])} damages",
            fontsize=16,
            fontweight='bold'
        )
        plt.tight_layout()
        
        # Save if requested
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"✅ Visualization saved to: {save_path}")
        
        plt.show()
        
        return img
    
    def get_patient_report(self, detections):
        """
        Generate a human-readable patient report
        
        Args:
            detections: Output from detect() method
            
        Returns:
            Formatted report string
        """
        report = []
        report.append("=" * 60)
        report.append("           DENTAL DETECTION REPORT")
        report.append("=" * 60)
        report.append("")
        
        # Teeth summary
        report.append(f"🦷 TEETH DETECTED: {len(detections['teeth'])}")
        report.append("-" * 60)
        
        if len(detections['teeth']) == 0:
            report.append("  No teeth detected in this image.")
        else:
            # Group by tooth type
            tooth_types = {}
            for tooth in detections['teeth']:
                tooth_type = tooth['class']
                if tooth_type not in tooth_types:
                    tooth_types[tooth_type] = []
                tooth_types[tooth_type].append(tooth)
            
            for tooth_type, teeth in tooth_types.items():
                report.append(f"\n  {tooth_type.upper()}S ({len(teeth)}):")
                for i, tooth in enumerate(teeth, 1):
                    report.append(
                        f"    {i}. {tooth['position'].capitalize()} {tooth['side']} "
                        f"(confidence: {tooth['confidence']:.1%})"
                    )
        
        report.append("")
        report.append("=" * 60)
        
        # Damage summary
        report.append(f"⚠️  DAMAGE DETECTED: {len(detections['damages'])}")
        report.append("-" * 60)
        
        if len(detections['damages']) == 0:
            report.append("  ✅ No damage detected! Teeth appear healthy.")
        else:
            for i, damage in enumerate(detections['damages'], 1):
                report.append(
                    f"  {i}. {damage['class'].upper()} on {damage['position']} {damage['side']} area"
                )
                report.append(f"     Confidence: {damage['confidence']:.1%}")
                report.append(f"     Location: {damage['bbox']}")
                report.append("")
        
        report.append("=" * 60)
        report.append("")
        report.append("NOTE: This is an AI-assisted screening tool.")
        report.append("Please consult a dental professional for diagnosis.")
        report.append("=" * 60)
        
        return "\n".join(report)
    
    def batch_detect(self, image_paths, conf_threshold=0.5):
        """
        Detect on multiple images
        
        Args:
            image_paths: List of image paths
            conf_threshold: Confidence threshold
            
        Returns:
            List of detection results
        """
        results = []
        for img_path in image_paths:
            print(f"Processing: {img_path}")
            detections = self.detect(img_path, conf_threshold)
            results.append(detections)
        return results

print("✅ DentalDetector class loaded successfully!")

---
## Part 5: Test the Model (Inference)

In [ ]:
# ============================================
# LOAD THE TRAINED MODEL
# ============================================

# Use the model you just trained
MODEL_PATH = "dental_detection_model.pt"  # or "Dental_Detection/YOLOv8_Training/weights/best.pt"

detector = DentalDetector(MODEL_PATH)

# ============================================
# TEST ON A SINGLE IMAGE
# ============================================

# Replace with your test image path
TEST_IMAGE = "path/to/your/test/image.jpg"  # UPDATE THIS!

# Perform detection
detections = detector.detect(
    TEST_IMAGE,
    conf_threshold=0.5  # Adjust confidence threshold (0.3-0.7 recommended)
)

# Print report
print(detector.get_patient_report(detections))

# Visualize results
detector.visualize(
    TEST_IMAGE,
    detections,
    save_path="detection_result.jpg"  # Optional: save visualization
)

---
## Part 6: Detailed Analysis

In [ ]:
# Access individual detections
print("\n" + "="*60)
print("DETAILED DETECTION BREAKDOWN")
print("="*60)

print(f"\n📊 Total Detections: {len(detections['teeth']) + len(detections['damages'])}")

# Teeth breakdown
print(f"\n🦷 Teeth ({len(detections['teeth'])})")
print("-" * 60)
for i, tooth in enumerate(detections['teeth'], 1):
    print(f"{i:2}. {tooth['full_label']:25} | Confidence: {tooth['confidence']:.2%}")
    print(f"    BBox: [{tooth['bbox'][0]:.1f}, {tooth['bbox'][1]:.1f}, {tooth['bbox'][2]:.1f}, {tooth['bbox'][3]:.1f}]")

# Damage breakdown
print(f"\n⚠️  Damage ({len(detections['damages'])})")
print("-" * 60)
if len(detections['damages']) == 0:
    print("✅ No damage detected!")
else:
    for i, damage in enumerate(detections['damages'], 1):
        print(f"{i:2}. {damage['class'].upper():15} | Position: {damage['position']:6} {damage['side']:5} | Confidence: {damage['confidence']:.2%}")
        print(f"    BBox: [{damage['bbox'][0]:.1f}, {damage['bbox'][1]:.1f}, {damage['bbox'][2]:.1f}, {damage['bbox'][3]:.1f}]")

# Statistics
print(f"\n📈 Statistics")
print("-" * 60)
if detections['teeth']:
    avg_tooth_conf = sum(t['confidence'] for t in detections['teeth']) / len(detections['teeth'])
    print(f"Average tooth detection confidence: {avg_tooth_conf:.2%}")

if detections['damages']:
    avg_damage_conf = sum(d['confidence'] for d in detections['damages']) / len(detections['damages'])
    print(f"Average damage detection confidence: {avg_damage_conf:.2%}")
    
    # Count by damage type
    damage_types = {}
    for damage in detections['damages']:
        dtype = damage['class']
        damage_types[dtype] = damage_types.get(dtype, 0) + 1
    
    print(f"\nDamage type distribution:")
    for dtype, count in damage_types.items():
        print(f"  {dtype.capitalize()}: {count}")

---
## Part 7: Batch Testing (Multiple Images)

In [ ]:
# Test on multiple images
import os
from glob import glob

# Get all test images
TEST_DIR = "yolo_dataset/test/images"  # Update path
test_images = glob(os.path.join(TEST_DIR, "*.jpg")) + glob(os.path.join(TEST_DIR, "*.png"))

print(f"Found {len(test_images)} test images")

# Process first 5 images (change as needed)
for img_path in test_images[:5]:
    print(f"\n{'='*60}")
    print(f"Processing: {os.path.basename(img_path)}")
    print(f"{'='*60}")
    
    detections = detector.detect(img_path, conf_threshold=0.5)
    
    # Quick summary
    print(f"Teeth: {len(detections['teeth'])}, Damage: {len(detections['damages'])}")
    
    # Visualize
    detector.visualize(img_path, detections)

---
## Part 8: Model Evaluation

Evaluate the model performance on the validation/test set

In [ ]:
from ultralytics import YOLO

# Load the trained model
model = YOLO(MODEL_PATH)

# Validate on test set
print("Evaluating model on test set...")
metrics = model.val(
    data=DATA_YAML,
    split='test',  # or 'val'
    conf=0.5,
    iou=0.45
)

# Print overall metrics
print("\n" + "="*60)
print("MODEL EVALUATION RESULTS")
print("="*60)
print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

print(f"\nInference speed: {metrics.speed['inference']:.1f}ms per image")

# Per-class metrics (if available)
print("\n" + "="*60)
print("PER-CLASS PERFORMANCE")
print("="*60)
print(f"{'Class':<15} {'Precision':<12} {'Recall':<12} {'mAP50':<12}")
print("-" * 60)

# Note: Access per-class metrics from the validation results
# This depends on the YOLOv8 version - check results object structure

---
## Part 9: Export Model for Production

Export to different formats for deployment

In [ ]:
# Export to ONNX (for faster inference, cross-platform)
model.export(format='onnx')
print("✅ Model exported to ONNX format")

# Export to TensorRT (for NVIDIA GPUs - fastest)
# model.export(format='engine')

# Export to TensorFlow Lite (for mobile)
# model.export(format='tflite')

# Export to CoreML (for iOS)
# model.export(format='coreml')

---
## Part 10: Flask API for Integration

Example API endpoint for your application

In [ ]:
# Save this code to a separate file (e.g., app.py) and run it

'''
from flask import Flask, request, jsonify
from PIL import Image
import io
import base64
import os

app = Flask(__name__)

# Initialize detector
detector = DentalDetector("dental_detection_model.pt")

@app.route('/detect', methods=['POST'])
def detect_dental():
    """
    API endpoint for dental detection
    
    Expected input (JSON):
    {
        "image": "base64_encoded_image_string"
    }
    
    Returns (JSON):
    {
        "teeth": [...],
        "damages": [...],
        "report": "...",
        "summary": {...}
    }
    """
    try:
        # Get image from request
        data = request.json
        image_data = base64.b64decode(data['image'])
        
        # Save temporarily
        img = Image.open(io.BytesIO(image_data))
        temp_path = "temp_upload.jpg"
        img.save(temp_path)
        
        # Detect
        detections = detector.detect(temp_path, conf_threshold=0.5)
        
        # Generate report
        report = detector.get_patient_report(detections)
        
        # Clean up
        os.remove(temp_path)
        
        # Return results
        return jsonify({
            'success': True,
            'teeth': detections['teeth'],
            'damages': detections['damages'],
            'report': report,
            'summary': {
                'total_teeth': len(detections['teeth']),
                'total_damages': len(detections['damages']),
                'has_damage': len(detections['damages']) > 0
            }
        })
    
    except Exception as e:
        return jsonify({
            'success': False,
            'error': str(e)
        }), 500

@app.route('/health', methods=['GET'])
def health_check():
    return jsonify({'status': 'healthy', 'model': 'loaded'})

if __name__ == '__main__':
    app.run(debug=True, host='0.0.0.0', port=5000)
'''

print("\n✅ Flask API code provided above")
print("   Save to app.py and run with: python app.py")
print("   API will be available at: http://localhost:5000")

---
## Summary & Next Steps

### What This Notebook Does:
1. ✅ Prepares dental dataset in YOLO format
2. ✅ Trains YOLOv8 to detect tooth types (incisors, canines, molars, premolars)
3. ✅ Trains YOLOv8 to detect damage (caries, cavity, decay, crack)
4. ✅ Provides position detection (upper/lower, left/right)
5. ✅ Generates patient reports
6. ✅ Includes API integration example

### Your Checklist:
- [ ] Prepare dataset with tooth type annotations
- [ ] Run Part 1 (dataset preparation)
- [ ] Run Part 3 (model training - 1-3 hours)
- [ ] Run Part 5 (test inference)
- [ ] Run Part 8 (evaluate performance)
- [ ] Integrate into your application using Part 10

### Tips for Better Results:
1. **More data is better**: Aim for 1000+ images
2. **Diverse images**: Different lighting, angles, cameras
3. **Accurate annotations**: Double-check your labels
4. **Higher epochs**: Try 150-200 epochs for better accuracy
5. **Larger model**: Use YOLOv8l or YOLOv8x if accuracy is critical

### Troubleshooting:
- **Low accuracy**: Need more training data or longer training
- **False positives**: Increase confidence threshold to 0.6-0.7
- **Out of memory**: Reduce batch size to 8 or 4
- **Slow training**: Use smaller model (yolov8n) or reduce image size

### Resources:
- Dataset labeling: https://roboflow.com or https://labelstud.io
- YOLOv8 docs: https://docs.ultralytics.com
- Dental datasets: Search Roboflow Universe, Kaggle

---

**Good luck with your dental detection project! 🦷**